# 🏏 Step 1: Initial Exploration & Baseline Experiments
### Target: Virat Kohli Boundary Prediction (IPL $\le$ 2023)

In this notebook, we explore the historical ball-by-ball IPL dataset, isolate **Virat Kohli's** batting deliveries, establish a binary target for boundaries (4s and 6s), analyze the severe class imbalance (~6:1), and benchmark a baseline **Random Forest Classifier** alongside naive sampling methods (**RandomOverSampler** and **RandomUnderSampler**).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Centralized bowler mapping (pace vs spin)
from src.bowler_mapping import map_bowler_type

## 1. Load Dataset & Filter for Virat Kohli
We load the raw IPL ball-by-ball dataset, isolate Virat Kohli's batting records up to 2023, filter out rare 7-run edge cases (overthrows/extras), and encode the binary boundary target (`runs_total > 3`).

In [2]:
# Load full ball-by-ball IPL dataset
df = pd.read_csv('IPL.csv', low_memory=False)

# Filter for Virat Kohli and matches up to 2023
df = df[df['batter'] == 'V Kohli'].copy()
df['date'] = pd.to_datetime(df['date'])
df = df[df['date'].dt.year <= 2023].copy()

# Classify match phase
def get_phase(over):
    if over <= 6:
        return "Powerplay"
    elif over <= 15:
        return "Middle"
    else:
        return "Death"

df['over_phase'] = df['over'].apply(get_phase)

# Drop rare 7-run balls and create binary boundary target
df = df.drop(df[df['runs_total'] == 7].index)
df['runs_total'] = np.where(df['runs_total'] > 3, 1, 0)

print(f"Total Kohli Deliveries Analyzed: {len(df)}")
df[['innings', 'over', 'ball', 'bat_pos', 'bowler', 'runs_batter', 'runs_total']].head()

Total Kohli Deliveries Analyzed: 5738


,innings,over,ball,bat_pos,bowler,runs_batter,runs_total
132,2,1,2,3,I Sharma,0,0
133,2,1,3,3,I Sharma,0,1
134,2,1,4,3,I Sharma,1,0
137,2,2,1,3,AB Dinda,0,0
138,2,2,2,3,AB Dinda,0,0


## 2. Innings Split & Class Imbalance Analysis
Innings 2 introduces chasing dynamics where `required_runs` comes into play. Below we split Innings 1 and Innings 2 and inspect the class distribution.

In [3]:
dfi1 = df[df['innings'] == 1].copy()
dfi2 = df[df['innings'] == 2].copy()

dfi1 = dfi1.drop(columns=['runs_target', 'innings'], errors='ignore')
dfi2['required_runs'] = dfi2['runs_target'] - dfi2['team_runs']
dfi2 = dfi2.drop(columns=['runs_target', 'team_runs', 'innings'], errors='ignore')

# Map bowlers to pace vs spin
dfi1['bowler'] = dfi1['bowler'].apply(map_bowler_type)
dfi2['bowler'] = dfi2['bowler'].apply(map_bowler_type)

# Calculate baseline progressive strike rate
dfi1['current_sr'] = (dfi1['batter_runs'] / dfi1['batter_balls'].replace(0, 1)) * 100
dfi2['current_sr'] = (dfi2['batter_runs'] / dfi2['batter_balls'].replace(0, 1)) * 100

print("--- Innings 1 Class Distribution ---")
print(dfi1['runs_total'].value_counts())
print("\nPercentage Breakdown:")
print(round(dfi1['runs_total'].value_counts(normalize=True) * 100, 2))

--- Innings 1 Class Distribution ---
runs_total
0    2680
1     462
Name: count, dtype: int64

Percentage Breakdown:
runs_total
0    85.3
1    14.7
Name: proportion, dtype: float64


## 3. Train / Test Split & Feature Preprocessing
We perform a 90/10 train/test split on Innings 1 data and one-hot encode the bowler category (`pace` vs `spin`).

In [4]:
feature_cols = [
    'ball', 'bat_pos', 'batter_runs', 'batter_balls',
    'bowler', 'team_runs', 'over', 'current_sr'
]

X = dfi1[feature_cols]
y = dfi1['runs_total']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), ["bowler"])],
    remainder="passthrough"
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"X_train shape: {X_train_proc.shape}, X_test shape: {X_test_proc.shape}")

X_train shape: (2827, 9), X_test shape: (315, 9)


## 4. Baseline Random Forest Model
We fit a baseline `RandomForestClassifier` with `class_weight='balanced'`.

In [5]:
rf_baseline = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)
rf_baseline.fit(X_train_proc, y_train)

y_pred_base = rf_baseline.predict(X_test_proc)
y_prob_base = rf_baseline.predict_proba(X_test_proc)[:, 1]

print("=== Baseline Random Forest Evaluation ===")
print("Accuracy :", round(accuracy_score(y_test, y_pred_base), 4))
print("Precision:", round(precision_score(y_test, y_pred_base), 4))
print("Recall   :", round(recall_score(y_test, y_pred_base), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_base), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob_base), 4))
print("PR-AUC   :", round(average_precision_score(y_test, y_prob_base), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_base))

=== Baseline Random Forest Evaluation ===
Accuracy : 0.8222
Precision: 0.4444
Recall   : 0.1481
F1 Score : 0.2222
ROC-AUC  : 0.7303
PR-AUC   : 0.3861
Confusion Matrix:
 [[251  10]
 [ 46   8]]


## 5. Resampling Experiments: RandomOverSampler vs RandomUnderSampler
To address the imbalanced target, we test naive oversampling (ROS) versus naive undersampling (RUS).

In [6]:
# 5a. RandomOverSampler
ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_train_proc, y_train)

rf_ros = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_ros.fit(X_ros, y_ros)
pred_ros = rf_ros.predict(X_test_proc)
prob_ros = rf_ros.predict_proba(X_test_proc)[:, 1]

print("=== Random Forest + RandomOverSampler ===")
print("Accuracy :", round(accuracy_score(y_test, pred_ros), 4))
print("Precision:", round(precision_score(y_test, pred_ros), 4))
print("Recall   :", round(recall_score(y_test, pred_ros), 4))
print("F1 Score :", round(f1_score(y_test, pred_ros), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_ros))

=== Random Forest + RandomOverSampler ===
Accuracy : 0.8032
Precision: 0.3889
Recall   : 0.2593
F1 Score : 0.3111
Confusion Matrix:
 [[239  22]
 [ 40  14]]


In [7]:
# 5b. RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_train_proc, y_train)

rf_rus = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_rus.fit(X_rus, y_rus)
pred_rus = rf_rus.predict(X_test_proc)
prob_rus = rf_rus.predict_proba(X_test_proc)[:, 1]

print("=== Random Forest + RandomUnderSampler ===")
print("Accuracy :", round(accuracy_score(y_test, pred_rus), 4))
print("Precision:", round(precision_score(y_test, pred_rus), 4))
print("Recall   :", round(recall_score(y_test, pred_rus), 4))
print("F1 Score :", round(f1_score(y_test, pred_rus), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_rus))

=== Random Forest + RandomUnderSampler ===
Accuracy : 0.6254
Precision: 0.2647
Recall   : 0.6667
F1 Score : 0.3789
Confusion Matrix:
 [[161 100]
 [ 18  36]]


### 💡 Key Takeaways from Step 1:
1. **Severe Imbalance (~6:1)**: Default models suffer on boundary recall (~14.8%) because predicting non-boundary yields >82% raw accuracy.
2. **Oversampling vs Undersampling**: ROS boosts F1 to ~0.31, while RUS boosts recall to ~66.7% at the cost of precision (26.5%).
3. **Next Step**: We need advanced temporal feature engineering (rolling strike rates, dot ball pressure, boundary droughts) and synthetic resampling (SMOTE). Let's move to **Step 2: Model Benchmarking**.